# Learned Stage-Wise Curvature Tuning on DermaMNIST

This notebook evaluates Learned Stage-Wise Curvature Tuning (SW-CT) on the DermaMNIST dataset using an ImageNet-pretrained ResNet-50.

DermaMNIST is a medical image-classification dataset containing dermatological lesion images. Its visual domain differs from the natural images used for ImageNet pretraining, making it useful for testing whether Learned SW-CT remains effective under substantial domain shift.

The experiments compare four conditions:

1. **Baseline** — the original pretrained ResNet with ReLU activations and a trained downstream classifier.
2. **Single-Parameter Curvature Tuning (S-CT)** — one global beta selected through validation-based exhaustive search.
3. **Learned SW-CT with fixed initialization** — four trainable stage-wise beta parameters initialized at 0.78.
4. **Learned SW-CT with random initialization** — four trainable stage-wise beta parameters independently initialized between 0.70 and 0.99.

Each condition is evaluated using seeds 42, 43, and 44.

Experiments are conducted for both 20 and 30 epochs to determine whether additional optimization improves performance or changes the learned stage-wise beta values.

Wall-clock execution time is also recorded to compare the computational cost of Baseline, S-CT, and Learned SW-CT.

In [ ]:
!nvidia-smi

In [ ]:
%cd /content

!git clone -b stage-wise-ct \
    https://github.com/Syed1611/curvature-tuning-research.git

%cd /content/curvature-tuning-research/src/curvature-tuning

!git branch --show-current
!git log -1 --oneline

## Environment Setup

The repository dependencies are installed before running the experiments.

After installation, the Colab runtime should be restarted so that the required package versions are loaded correctly.

In [ ]:
%pip install -q -r requirements.txt
%pip install --force-reinstall --no-cache-dir "numpy==1.26.3"

In [ ]:
%cd /content/curvature-tuning-research/src/curvature-tuning

import os
import torch

os.environ["WANDB_MODE"] = "disabled"

print("Working directory:", os.getcwd())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Arabic Digits Dataset

Before running the experiments, the Arabic Digits data loaders are checked to confirm that the downstream dataset can be loaded correctly.

The same dataset split and random seed protocol are used for Baseline, S-CT, and Learned SW-CT so that the methods are compared under the same conditions.

In [ ]:
from utils.data import get_data_loaders

train_loader, test_loader, val_loader = get_data_loaders(
    "imagenet_to_medmnist/dermamnist",
    train_batch_size=32,
    test_batch_size=800,
    seed=42,
)

print("Train samples:", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels:", labels[:10])

# ResNet-50 Experiments

ResNet-50 is evaluated first because it was used during the initial development of Learned SW-CT.

The experiment compares Baseline, S-CT, fixed-initialization SW-CT, and random-initialization SW-CT.

All methods use the same ImageNet-pretrained backbone, Arabic Digits dataset split, experimental seeds, and downstream training protocol.

## ResNet-50 — 20-Epoch Evaluation

The first evaluation uses 20 training epochs.

This provides the primary reference experiment and maintains compatibility with the earlier Learned SW-CT work.

### Methods

**Baseline**

The pretrained ResNet-50 retains its original ReLU activations. The backbone is frozen and only the downstream classifier is trained.

**S-CT**

The original Single-Parameter Curvature Tuning method searches over a single beta shared throughout the network. Candidate beta values from 0.70 to 1.00 are evaluated using validation performance, and the best candidate is evaluated on the test set.

The recorded S-CT execution time includes the complete beta search rather than only the final selected beta.

**Learned SW-CT — Fixed Initialization**

Four trainable beta parameters are assigned to the four ResNet stages. All four begin at beta = 0.78.

This preserves the initialization used in the original Learned SW-CT experiments.

**Learned SW-CT — Random Initialization**

Four trainable stage-wise beta parameters are initialized independently between 0.70 and 0.99.

The random initialization is generated deterministically from the experiment seed. This allows the experiment to test whether Learned SW-CT depends on the fixed 0.78 starting value while remaining reproducible.

In [ ]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 800 \
    --epochs 20

In [ ]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 800 \
    --epochs 20

In [ ]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 800 \
    --epochs 20

### Learned SW-CT with Fixed Beta Initialization

The original Learned SW-CT configuration is first reproduced using the fixed initialization:

\[[\beta_1,\beta_2,\beta_3,\beta_4]=[0.78,0.78,0.78,0.78].\]

The initial value is only a starting point. During training, the four beta parameters are optimized independently together with the downstream classifier.

These runs provide the reference against which random initialization will be compared.

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 800 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 800 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 800 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

### Learned SW-CT with Random Beta Initialization

The fixed beta = 0.78 initialization may raise the question of whether Learned SW-CT performs well only because its starting point was informed by the earlier S-CT experiments.

To test this dependency, each of the four stage-wise beta parameters is now initialized independently from a uniform distribution between 0.70 and 0.99.

The initialization is reproducible: the random generator is seeded using the experiment seed. Therefore, each seed always receives the same initial beta vector.

If random initialization produces similar performance to the fixed 0.78 experiment, this provides evidence that the Learned SW-CT result is not dependent on a single hand-selected initialization.

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 800 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 800 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 800 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

# ResNet-50 — 30-Epoch Evaluation

The same experiments are now repeated for 30 epochs.

The purpose of increasing the training budget is to determine whether the 20-epoch results are limited by insufficient optimization.

If a method improves substantially with 30 epochs, this suggests that additional optimization is useful. If performance remains similar, it indicates that the method has largely stabilized within the original 20-epoch training budget.

All other experimental settings remain unchanged.

For random-initialization SW-CT, the same seed produces exactly the same initial four-beta vector in both the 20-epoch and 30-epoch experiments. Therefore, initialization is held constant while the training duration changes.

Execution time is again recorded to measure the computational cost of extending training from 20 to 30 epochs.

In [ ]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 800 \
    --epochs 30

In [ ]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 800 \
    --epochs 30

In [ ]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 800 \
    --epochs 30

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 800 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 800 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 800 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 800 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 800 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 800 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30